# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/redey/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 22:28:41.021439: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 22:28:41.023627: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 22:28:41.069952: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 22:28:41.069995: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 22:28:41.070042: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"]).values.astype(np.float32)
y = df["Class"].values.astype(np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Classes: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Training samples: 124
Test samples: 54


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaled training mean, first 5 features:", np.round(X_train_scaled.mean(axis=0)[:5], 4))
print("Scaled training std, first 5 features:", np.round(X_train_scaled.std(axis=0)[:5], 4))

Scaled training mean, first 5 features: [ 0. -0.  0. -0.  0.]
Scaled training std, first 5 features: [1. 1. 1. 1. 1.]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation="relu", input_shape=(num_features,)),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 1s 13ms/step - loss: 1.0150 - accuracy: 0.5051 - val_loss: 0.7687 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 4ms/step - loss: 0.7210 - accuracy: 0.8081 - val_loss: 0.5665 - val_accuracy: 0.8800
Epoch 3/20
13/13 [==============================] - 0s 4ms/step - loss: 0.5244 - accuracy: 0.9192 - val_loss: 0.4190 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 4ms/step - loss: 0.3774 - accuracy: 0.9596 - val_loss: 0.3074 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 4ms/step - loss: 0.2689 - accuracy: 0.9697 - val_loss: 0.2286 - val_accuracy: 0.9200
Epoch 6/20
13/13 [==============================] - 0s 4ms/step - loss: 0.1950 - accuracy: 0.9798 - val_loss: 0.1838 - val_accuracy: 0.9200
Epoch 7/20
13/13 [==============================] - 0s 4ms/step - loss: 0.1436 - accuracy: 0.9798 - val_loss: 0.1530 - val_accuracy: 0.9600
Epoch 8/20
13/13 [=

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)

y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(f"Baseline DNN training accuracy: {train_acc:.4f}")
print(f"Baseline DNN test accuracy: {test_acc:.4f}")

print("\nBaseline classification report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

print("Baseline confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Baseline DNN training accuracy: 0.9919
Baseline DNN test accuracy: 0.9815

Baseline classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Baseline confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
import os

def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024.0

base_converter = tf.lite.TFLiteConverter.from_keras_model(model)
base_tflite = base_converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(base_tflite)

print(f"Float32 baseline TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp1l5bl7bs/assets


INFO:tensorflow:Assets written to: /tmp/tmp1l5bl7bs/assets


Float32 baseline TFLite model size: 14.07 KB


2026-05-20 22:29:11.747900: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:29:11.747951: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:29:11.748248: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp1l5bl7bs
2026-05-20 22:29:11.749209: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:29:11.749223: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp1l5bl7bs
2026-05-20 22:29:11.752689: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 22:29:11.753484: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:29:11.787403: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp1l5bl7bs
2026-05

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]

def _quantize_input(sample, input_detail):
    """
    Quantize a float32 input sample if the TFLite model expects int8/uint8 input.
    Otherwise, return the sample as float32.
    """
    input_dtype = input_detail["dtype"]

    if input_dtype == np.float32:
        return sample.astype(np.float32)

    scale, zero_point = input_detail["quantization"]

    if scale == 0:
        return sample.astype(input_dtype)

    quantized_sample = sample / scale + zero_point
    quantized_sample = np.round(quantized_sample)

    if input_dtype == np.int8:
        quantized_sample = np.clip(quantized_sample, -128, 127)
    elif input_dtype == np.uint8:
        quantized_sample = np.clip(quantized_sample, 0, 255)

    return quantized_sample.astype(input_dtype)

def _dequantize_output(output, output_detail):
    """
    Dequantize int8/uint8 TFLite output back to float values.
    If output is already float32, return it unchanged.
    """
    output_dtype = output_detail["dtype"]

    if output_dtype == np.float32:
        return output.astype(np.float32)

    scale, zero_point = output_detail["quantization"]

    if scale == 0:
        return output.astype(np.float32)

    return scale * (output.astype(np.float32) - zero_point)
    
def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(X_test.shape[0]):
        sample = X_test[i:i + 1].astype(np.float32)
        sample = _quantize_input(sample, input_detail)

        interpreter.set_tensor(input_detail["index"], sample)
        interpreter.invoke()

        output = interpreter.get_tensor(output_detail["index"])
        output = _dequantize_output(output, output_detail)
        y_pred.append(int(np.argmax(output, axis=1)[0]))

    y_true = np.argmax(y_test_cat, axis=1)
    # Step 4: Report results.
    
    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    accuracy = accuracy_score(y_true, y_pred)
    size_kb = file_size_kb(filename)

    print(f"\n{quant_type.upper()} TFLite model size: {size_kb:.2f} KB")
    print(f"{quant_type.upper()} TFLite accuracy: {accuracy:.4f}")
    print(f"\n{quant_type.upper()} classification report:")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))
    print(f"{quant_type.upper()} confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "quant_type": quant_type,
        "filename": filename,
        "size_kb": size_kb,
        "accuracy": accuracy,
        "y_true": y_true,
        "y_pred": np.array(y_pred)
    }

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quant_results = {}

quant_results["int8"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="int8",
    filename="model_int8.tflite"
)

quant_results["float16"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="float16",
    filename="model_float16.tflite"
)

quant_results["dynamic"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="dynamic",
    filename="model_dynamic.tflite"
)

INFO:tensorflow:Assets written to: /tmp/tmplqqr9wov/assets


INFO:tensorflow:Assets written to: /tmp/tmplqqr9wov/assets
/home/redey/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 22:29:22.213338: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:29:22.213385: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:29:22.213536: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmplqqr9wov
2026-05-20 22:29:22.214648: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:29:22.214669: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmplqqr9wov
2026-05-20 22:29:22.216834: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202


INT8 TFLite model size: 5.74 KB
INT8 TFLite accuracy: 0.9815

INT8 classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

INT8 confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpmdd_1bp9/assets


INFO:tensorflow:Assets written to: /tmp/tmpmdd_1bp9/assets
2026-05-20 22:29:22.697891: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:29:22.697975: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:29:22.698125: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmdd_1bp9
2026-05-20 22:29:22.698722: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:29:22.698734: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpmdd_1bp9
2026-05-20 22:29:22.700419: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:29:22.735882: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpmdd_1bp9
2026-05-20 22:29:22.744637: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.95 KB
FLOAT16 TFLite accuracy: 0.9815

FLOAT16 classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

FLOAT16 confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmp87dhz53r/assets


INFO:tensorflow:Assets written to: /tmp/tmp87dhz53r/assets
2026-05-20 22:29:23.204157: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:29:23.204218: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:29:23.204353: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp87dhz53r
2026-05-20 22:29:23.204864: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:29:23.204878: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp87dhz53r
2026-05-20 22:29:23.206356: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:29:23.232369: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp87dhz53r
2026-05-20 22:29:23.241129: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 8.17 KB
DYNAMIC TFLite accuracy: 0.9815

DYNAMIC classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

DYNAMIC confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
pruning_epochs = 10
pruning_batch_size = 8
steps_per_epoch = int(np.ceil((X_train_scaled.shape[0] * 0.8) / pruning_batch_size))
end_step = steps_per_epoch * pruning_epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.50,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step
)

print("Steps per epoch:", steps_per_epoch)
print("Pruning end step:", end_step)

Steps per epoch: 13
Pruning end step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation="relu", input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation="relu"),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(num_classes, activation="softmax"),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

pruning_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 2s 13ms/step - loss: 0.9983 - accuracy: 0.4848 - val_loss: 0.8670 - val_accuracy: 0.7200
Epoch 2/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6829 - accuracy: 0.8081 - val_loss: 0.6340 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.4876 - accuracy: 0.9091 - val_loss: 0.4718 - val_accuracy: 0.8800
Epoch 4/10
13/13 [==============================] - 0s 4ms/step - loss: 0.3496 - accuracy: 0.9495 - val_loss: 0.3464 - val_accuracy: 0.8800
Epoch 5/10
13/13 [==============================] - 0s 4ms/step - loss: 0.2502 - accuracy: 0.9899 - val_loss: 0.2621 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 4ms/step - loss: 0.1824 - accuracy: 0.9899 - val_loss: 0.2065 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 4ms/step - loss: 0.1345 - accuracy: 0.9899 - val_loss: 0.1653 - val_accuracy: 0.9200
Epoch 8/10
13/13 [=

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

pruned_converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
pruned_tflite = pruned_converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(pruned_tflite)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp_glk4b7v/assets


INFO:tensorflow:Assets written to: /tmp/tmp_glk4b7v/assets


Pruned TFLite model size: 14.14 KB


2026-05-20 22:29:37.689601: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:29:37.689662: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:29:37.689842: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_glk4b7v
2026-05-20 22:29:37.690453: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:29:37.690467: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp_glk4b7v
2026-05-20 22:29:37.691860: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:29:37.706631: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp_glk4b7v
2026-05-20 22:29:37.711914: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 22071 m

In [22]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
stripped_pruned_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
pruned_train_loss, pruned_train_acc = stripped_pruned_model.evaluate(
    X_train_scaled,
    y_train_cat,
    verbose=0
)
pruned_test_loss, pruned_test_acc = stripped_pruned_model.evaluate(
    X_test_scaled,
    y_test_cat,
    verbose=0
)

pruned_probs = stripped_pruned_model.predict(X_test_scaled, verbose=0)
pruned_pred = np.argmax(pruned_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(f"Pruned model training accuracy: {pruned_train_acc:.4f}")
print(f"Pruned model test accuracy: {pruned_test_acc:.4f}")

print("\nPruned model classification report:")
print(classification_report(y_true, pruned_pred, target_names=wine.target_names))

print("Pruned model confusion matrix:")
print(confusion_matrix(y_true, pruned_pred))

Pruned model training accuracy: 0.9758
Pruned model test accuracy: 0.9630

Pruned model classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       0.95      0.95      0.95        21
     class_2       0.93      0.93      0.93        15

    accuracy                           0.96        54
   macro avg       0.96      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Pruned model confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  1 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [23]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation="relu", input_shape=(num_features,)),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax")
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_preds_soft = model.predict(X_train_scaled, verbose=0)

print("Teacher soft labels shape:", teacher_preds_soft.shape)
print("First teacher soft label:", np.round(teacher_preds_soft[0], 4))

Teacher soft labels shape: (124, 3)
First teacher soft label: [0.9978 0.0011 0.0011]


In [25]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
alpha = 0.5
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1).astype(np.float32)

print("Combined distillation label shape:", y_train_combined.shape)

def distillation_loss(y_true_combined, y_pred):
    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1.0 - alpha) * soft_loss

Combined distillation label shape: (124, 6)


In [26]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=distillation_loss,
    metrics=["accuracy"]
)

student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 14ms/step - loss: 0.9175 - accuracy: 0.6263 - val_loss: 0.8162 - val_accuracy: 0.7600
Epoch 2/10
13/13 [==============================] - 0s 4ms/step - loss: 0.7551 - accuracy: 0.7980 - val_loss: 0.6860 - val_accuracy: 0.8000
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6310 - accuracy: 0.9293 - val_loss: 0.5774 - val_accuracy: 0.8400
Epoch 4/10
13/13 [==============================] - 0s 4ms/step - loss: 0.5178 - accuracy: 0.9495 - val_loss: 0.4792 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 4ms/step - loss: 0.4175 - accuracy: 0.9596 - val_loss: 0.3945 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 4ms/step - loss: 0.3337 - accuracy: 0.9798 - val_loss: 0.3258 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 4ms/step - loss: 0.2675 - accuracy: 0.9798 - val_loss: 0.2680 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [27]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
kd_converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
kd_tflite = kd_converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(kd_tflite)

print(f"Knowledge-distilled student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpwvacwd5t/assets


INFO:tensorflow:Assets written to: /tmp/tmpwvacwd5t/assets


Knowledge-distilled student TFLite model size: 6.10 KB


2026-05-20 22:31:50.105618: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:31:50.105668: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:31:50.105845: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpwvacwd5t
2026-05-20 22:31:50.106561: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:31:50.106574: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpwvacwd5t
2026-05-20 22:31:50.108697: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:31:50.138481: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpwvacwd5t
2026-05-20 22:31:50.147207: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 41361 m

In [28]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
student_probs = student_model.predict(X_test_scaled, verbose=0)
student_pred = np.argmax(student_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

student_accuracy = accuracy_score(y_true, student_pred)

print(f"Knowledge-distilled student accuracy: {student_accuracy:.4f}")

print("\nKnowledge-distilled student classification report:")
print(classification_report(y_true, student_pred, target_names=wine.target_names))

print("Knowledge-distilled student confusion matrix:")
print(confusion_matrix(y_true, student_pred))

Knowledge-distilled student accuracy: 0.9815

Knowledge-distilled student classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Knowledge-distilled student confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [29]:
# Part (e): Further reduce model size by training a very small student model
# and then applying full integer quantization.
#
# Strategy:
# 1. Use fewer hidden units than the KD student from part (d).
# 2. Train using the same output-based distillation loss.
# 3. Convert the tiny student using full integer quantization.
#
# This combines architecture reduction and quantization.

tiny_student_model = Sequential([
    Dense(12, activation="relu", input_shape=(num_features,)),
    Dense(6, activation="relu"),
    Dense(num_classes, activation="softmax")
])

tiny_student_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=distillation_loss,
    metrics=["accuracy"]
)

tiny_history = tiny_student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

tiny_probs = tiny_student_model.predict(X_test_scaled, verbose=0)
tiny_pred = np.argmax(tiny_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

tiny_accuracy = accuracy_score(y_true, tiny_pred)

print(f"Tiny distilled student float32 accuracy before quantization: {tiny_accuracy:.4f}")
print("\nTiny distilled student float32 classification report:")
print(classification_report(y_true, tiny_pred, target_names=wine.target_names))
print("Tiny distilled student float32 confusion matrix:")
print(confusion_matrix(y_true, tiny_pred))

tiny_int8_result = quantize_and_evaluate(
    tiny_student_model,
    X_test_scaled,
    y_test_cat,
    quant_type="int8",
    filename="model_tiny_kd_int8.tflite"
)

# Summary table comparing all saved TFLite models.
saved_models = [
    ("Baseline float32", "model_base.tflite", test_acc),
    ("Dynamic range quantized", "model_dynamic.tflite", quant_results["dynamic"]["accuracy"]),
    ("Full integer int8 quantized", "model_int8.tflite", quant_results["int8"]["accuracy"]),
    ("Float16 quantized", "model_float16.tflite", quant_results["float16"]["accuracy"]),
    ("Pruned float32", "model_pruned.tflite", pruned_test_acc),
    ("KD student float32", "model_kd.tflite", student_accuracy),
    ("Tiny KD student int8", "model_tiny_kd_int8.tflite", tiny_int8_result["accuracy"]),
]

comparison_rows = []
for model_name, filename, accuracy in saved_models:
    comparison_rows.append({
        "Model": model_name,
        "File": filename,
        "Size KB": file_size_kb(filename),
        "Accuracy": accuracy
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Size KB").reset_index(drop=True)
print("\nModel size and accuracy comparison:")
display(comparison_df)

best_size_row = comparison_df.iloc[0]
print(
    f"Smallest model: {best_size_row['Model']} "
    f"({best_size_row['Size KB']:.2f} KB, accuracy={best_size_row['Accuracy']:.4f})"
)

Epoch 1/30
13/13 [==============================] - 1s 14ms/step - loss: 1.5093 - accuracy: 0.2424 - val_loss: 1.6621 - val_accuracy: 0.3200
Epoch 2/30
13/13 [==============================] - 0s 4ms/step - loss: 1.3819 - accuracy: 0.2525 - val_loss: 1.4956 - val_accuracy: 0.4000
Epoch 3/30
13/13 [==============================] - 0s 5ms/step - loss: 1.2813 - accuracy: 0.2727 - val_loss: 1.3631 - val_accuracy: 0.4000
Epoch 4/30
13/13 [==============================] - 0s 4ms/step - loss: 1.1943 - accuracy: 0.3636 - val_loss: 1.2601 - val_accuracy: 0.4800
Epoch 5/30
13/13 [==============================] - 0s 4ms/step - loss: 1.1343 - accuracy: 0.4242 - val_loss: 1.1677 - val_accuracy: 0.4800
Epoch 6/30
13/13 [==============================] - 0s 4ms/step - loss: 1.0740 - accuracy: 0.5152 - val_loss: 1.0969 - val_accuracy: 0.4800
Epoch 7/30
13/13 [==============================] - 0s 4ms/step - loss: 1.0320 - accuracy: 0.5455 - val_loss: 1.0310 - val_accuracy: 0.4800
Epoch 8/30
13/13 [=

INFO:tensorflow:Assets written to: /tmp/tmpgd3ryx2f/assets
/home/redey/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 22:32:00.608715: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:32:00.608767: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



INT8 TFLite model size: 2.83 KB
INT8 TFLite accuracy: 0.9444

INT8 classification report:
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        18
     class_1       0.88      1.00      0.93        21
     class_2       1.00      0.93      0.97        15

    accuracy                           0.94        54
   macro avg       0.96      0.94      0.95        54
weighted avg       0.95      0.94      0.94        54

INT8 confusion matrix:
[[16  2  0]
 [ 0 21  0]
 [ 0  1 14]]

Model size and accuracy comparison:


2026-05-20 22:32:00.608941: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgd3ryx2f
2026-05-20 22:32:00.610213: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:32:00.610258: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpgd3ryx2f
2026-05-20 22:32:00.612908: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:32:00.649581: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpgd3ryx2f
2026-05-20 22:32:00.659216: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 50272 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


,Model,File,Size KB,Accuracy
0,Tiny KD student int8,model_tiny_kd_int8.tflite,2.828125,0.944444
1,Full integer int8 quantized,model_int8.tflite,5.742188,0.981481
2,KD student float32,model_kd.tflite,6.101562,0.981481
3,Dynamic range quantized,model_dynamic.tflite,8.171875,0.981481
4,Float16 quantized,model_float16.tflite,8.945312,0.981481
5,Baseline float32,model_base.tflite,14.070312,0.981481
6,Pruned float32,model_pruned.tflite,14.140625,0.962963


Smallest model: Tiny KD student int8 (2.83 KB, accuracy=0.9444)


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
